# DDVC YouCook2: localization-quality ranking

Train a lightweight quality head alongside original DDVC, then compare original/quality ranking on identical saved candidates. Randomly initialized GPT-2 decoder and global caption visual attention are retained. No temporal fusion, bidirectional interaction, semantic matching or narrative memory is enabled.

**Default: subset 30% train and validation, 15 epochs, seed 777, batch 1, 50 queries, one T4.** GPU + Internet must be enabled. Data downloads automatically; no Kaggle Input dataset is required. This notebook trains DDVC from scratch using pre-extracted CLIP features, not raw video encoders. Improvement in CIDEr is an experimental question, not guaranteed.


In [ ]:
from pathlib import Path
import os, sys, json, random, subprocess, time, collections
import torch

DATA_ROOT = Path('/kaggle/working/youcook2_ddvc_data')
RUN_MODE = 'subset'  # 'smoke' = 1%, 'subset' = 30%, 'full' = all feature-matched videos
SMOKE_FRACTION = 0.01
SUBSET_FRACTION = 0.30
if RUN_MODE not in ('smoke', 'subset', 'full'):
    raise ValueError("RUN_MODE must be 'smoke', 'subset', or 'full'")
ACTIVE_FRACTION = SMOKE_FRACTION if RUN_MODE == 'smoke' else (SUBSET_FRACTION if RUN_MODE == 'subset' else 1.0)
EPOCHS = 2 if RUN_MODE == 'smoke' else (15 if RUN_MODE == 'subset' else 25)
SEED = 777
ENABLE_QUALITY = True  # False = separate no-head training control
QUALITY_RANK_WEIGHT = 1.0  # 0 = original ranking; paired comparison runs after training
QUALITY_LOSS_COEF = 1.0
if QUALITY_RANK_WEIGHT < 0 or QUALITY_LOSS_COEF <= 0:
    raise ValueError('Quality ranking weight must be >= 0 and loss coefficient > 0')
EXPERIMENT = 'quality' if ENABLE_QUALITY else 'baseline'
RESUME_RUN_ID = ''  # leave blank for a fresh run; set to this run ID to resume model-last.pth
RUN_ID = RESUME_RUN_ID or f'yc2_{RUN_MODE}_{EXPERIMENT}'
REPO = Path('/kaggle/working/DDVC_quality_ranking')
SAVE_DIR = 'save_quality_ranking'
LOG_PATH = Path(f'/kaggle/working/ddvc_{EXPERIMENT}_train.log')
SOURCE_COMMIT = '54adf8dbbcfe6bf7cc8bff6b6e0193eb257e55f4'
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings.'
print('Mode:', RUN_MODE, '| fraction per split:', f'{ACTIVE_FRACTION:.0%}', '| epochs:', EPOCHS)
print('GPU:', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GiB')
print('Data will be downloaded into', DATA_ROOT)


## Clone repo, cài runtime, build CUDA op

Bản upstream đọc GPT-2 tokenizer từ đường dẫn riêng của tác giả; notebook chuyển sang tên model public. Extension Deformable Attention cần build cho compute capability 7.5 của T4.

In [ ]:
REPO_URL = 'https://github.com/thloc1212/DDVC.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'exp/localization-quality-ranking', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
# Clear Kaggle-only compatibility edits from an earlier interrupted run before switching revisions.
subprocess.run(['git', '-C', str(REPO), 'reset', '--hard'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
print('DDVC source commit:', actual_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML', 'tensorboardX', 'colorlog', 'ftfy', 'regex', 'gdown', 'pycocotools', 'setuptools<81', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '--no-deps', 'git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1'], check=True)
# Patch deprecated PyTorch APIs used by this legacy CUDA extension.
ops_dir = REPO/'ddvc'/'ops'
cuda_src = ops_dir/'src'/'cuda'/'ms_deform_attn_cuda.cu'
cuda_text = cuda_src.read_text()
legacy, fixed = 'AT_DISPATCH_FLOATING_TYPES(value.type(),', 'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(),'
if legacy in cuda_text:
    cuda_text = cuda_text.replace(legacy, fixed)
elif fixed not in cuda_text:
    raise RuntimeError(f'Could not apply PyTorch compatibility patch to {cuda_src}')
cuda_text = cuda_text.replace('.data<', '.data_ptr<')
cuda_src.write_text(cuda_text)

# Discard partial objects from an interrupted compile and avoid Ninja launching
# more compiler processes than Kaggle's small CPU/RAM allocation can sustain.
import shutil
shutil.rmtree(ops_dir/'build', ignore_errors=True)
env = os.environ.copy()
env.update({'TORCH_CUDA_ARCH_LIST':'7.5', 'MAX_JOBS':'2'})
print('Building the CUDA extension with MAX_JOBS=2 ...', flush=True)
subprocess.run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops_dir, env=env, check=True, timeout=1200)
print('CUDA extension build and install completed.', flush=True)


## Tải annotation chính thức và CLIP features của DDVC

Không cần chọn hay gắn dataset trong Kaggle Input. Bật Internet để notebook tải annotation YouCook2 chính thức và file CLIP ViT-L/14 mà repo DDVC cung cấp. Sau đó notebook đối chiếu ID video, in số liệu độ phủ, và chỉ loại video không có feature tương ứng.


In [ ]:
import json, tarfile, requests
from pathlib import Path

DATA_ROOT.mkdir(parents=True, exist_ok=True)


def download_file(url, destination):
    destination=Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    print('Downloading:', url)
    partial=destination.with_name(destination.name+'.part')
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with partial.open('wb') as f:
            for chunk in response.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    f.write(chunk)
    partial.replace(destination)
    return destination


def convert_youcook2(raw):
    database=raw.get('database', raw)
    splits={'training':{}, 'validation':{}}
    for video_id, item in database.items():
        subset=item.get('subset')
        if subset not in splits:
            continue
        annotations=[a for a in item.get('annotations', [])
                     if a.get('sentence') and isinstance(a.get('segment'), (list, tuple)) and len(a['segment']) == 2]
        if not annotations:
            continue
        splits[subset][video_id]={
            'duration': float(item['duration']),
            'sentences': [a['sentence'] for a in annotations],
            'timestamps': [[float(a['segment'][0]), float(a['segment'][1])] for a in annotations],
        }
    return splits['training'], splits['validation']


# Official YouCook2 combined train/validation annotations.
try:
    ann_tar=download_file(
        'https://youcook2.eecs.umich.edu/static/YouCookII/youcookii_annotations_trainval.tar.gz',
        DATA_ROOT/'youcookii_annotations_trainval.tar.gz')
    with tarfile.open(ann_tar, 'r:gz') as archive:
        members=[m for m in archive.getmembers() if Path(m.name).name == 'youcookii_annotations_trainval.json' and m.isfile()]
        if len(members) != 1:
            raise FileNotFoundError('Official annotation archive did not contain exactly one youcookii_annotations_trainval.json')
        with archive.extractfile(members[0]) as f:
            raw_annotations=json.load(f)
except requests.exceptions.SSLError:
    # Kaggle mirror of the same official annotation file, used only if the UMich TLS chain fails.
    print('Official host SSL verification failed; using the Kaggle mirror of the official annotation file.')
    ann_json=download_file(
        'https://www.kaggle.com/api/v1/datasets/download/ravishkumarcs23m055/annotation-youcook2/youcookii_annotations_trainval.json',
        DATA_ROOT/'youcookii_annotations_trainval.json')
    with ann_json.open(encoding='utf-8') as f:
        raw_annotations=json.load(f)

train_all,val_all=convert_youcook2(raw_annotations)
del raw_annotations
assert train_all and val_all, 'Official YouCook2 train/validation annotations did not load.'
print('Official annotations:',len(train_all),'train videos,',len(val_all),'validation videos')

# DDVC's pre-extracted CLIP ViT-L/14 features (linked from its official repository README).
feature_src=DATA_ROOT/'clipvitl14.pth'
if not feature_src.exists() or feature_src.stat().st_size < 100*1024*1024:
    if feature_src.exists(): feature_src.unlink()
    import gdown
    downloaded=gdown.download(
        id='17H_lxSKFve57kHpkAD7pcYS7ijHIy4-M',
        output=str(feature_src), quiet=False)
    if downloaded is None or not feature_src.exists() or feature_src.stat().st_size < 100*1024*1024:
        raise RuntimeError('CLIP feature download failed or is incomplete. Enable Kaggle Internet and retry; expected the DDVC clipvitl14.pth file.')
print('DDVC CLIP features:',feature_src,round(feature_src.stat().st_size/1024**2),'MiB')

for split,data in [('train',train_all),('validation',val_all)]:
    assert isinstance(data,dict) and data, f'{split} annotation split is empty'
    assert {'duration','sentences','timestamps'} <= set(next(iter(data.values()))), f'{split}.json has an unexpected format'
try:
    feature_map=torch.load(feature_src,map_location='cpu',weights_only=False)
except TypeError:
    feature_map=torch.load(feature_src,map_location='cpu')
if 'features' in feature_map and isinstance(feature_map['features'],dict):
    feature_map=feature_map['features']
assert isinstance(feature_map,dict), f'Expected a video-ID to feature mapping in {feature_src}'
feature_ids=set(feature_map)


def choose(data,n,seed,split_name):
    valid_ids=[k for k,v in data.items() if k in feature_ids and v.get('sentences') and len(v['sentences'])==len(v['timestamps'])]
    missing_ids=set(data)-feature_ids
    random.Random(seed).shuffle(valid_ids)
    if not valid_ids:
        raise ValueError(f'No {split_name} videos match the downloaded CLIP feature IDs.')
    selected_ids=valid_ids if n is None else valid_ids[:min(n,len(valid_ids))]
    selected={k:data[k] for k in selected_ids}
    events_all=sum(len(v['sentences']) for v in data.values())
    events_selected=sum(len(v['sentences']) for v in selected.values())
    print(f'{split_name}: {len(data):,} official annotated videos / {len(valid_ids):,} with features / {len(selected):,} used; {events_all:,} annotations / {events_selected:,} used')
    print(f'  Missing feature for {len(missing_ids):,} annotated videos; examples: {sorted(missing_ids)[:8]}')
    print(f'  Feature-matched but omitted by subset limit: {len(set(valid_ids)-set(selected_ids)):,}')
    return selected

train_available=sum(k in feature_ids for k in train_all)
val_available=sum(k in feature_ids for k in val_all)
TRAIN_VIDEOS=None if RUN_MODE=='full' else max(1, round(train_available*ACTIVE_FRACTION))
VAL_VIDEOS=None if RUN_MODE=='full' else max(1, round(val_available*ACTIVE_FRACTION))
train=choose(train_all,TRAIN_VIDEOS,SEED,'train')
val=choose(val_all,VAL_VIDEOS,SEED+1,'validation')
assert not (set(train)&set(val)), 'Train/validation video IDs overlap.'

REPO_DATA=REPO/'data'/'yc2'
ANNO=REPO_DATA/'captiondata'
ANNO.mkdir(parents=True,exist_ok=True)
train_json,val_json=ANNO/'train.json',ANNO/'val.json'
train_json.write_text(json.dumps(train),encoding='utf-8')
val_json.write_text(json.dumps(val),encoding='utf-8')

# Upstream paragraph evaluator expects video_id -> concatenated reference captions.
para_dir=ANNO/'para'
para_dir.mkdir(parents=True,exist_ok=True)
para_json=para_dir/'para_yc2_val.json'
para_refs={}
for vid,record in val.items():
    ordered=sorted(zip(record['timestamps'],record['sentences']),key=lambda pair:float(pair[0][0]))
    para_refs[vid]=' '.join(sentence.strip().rstrip('.')+'.' for _,sentence in ordered)
para_json.write_text(json.dumps(para_refs),encoding='utf-8')
print('Wrote full matched validation references:',para_json,'videos:',len(para_refs))

small={k:torch.as_tensor(feature_map[k]).detach().cpu().float() for k in set(train)|set(val)}
features_path=REPO_DATA/'features'/f'clipvitl14_{RUN_MODE}.pth'
features_path.parent.mkdir(parents=True,exist_ok=True)
torch.save(small,features_path)
del feature_map,small
print('Prepared selected data:',len(train),'train,',len(val),'validation videos; feature file:',features_path)


## CLIP retrieval memory for the selected training subset

The original DDVC retrieval mechanism is retained. The memory bank is built only from the selected training captions; validation captions are not included.


In [ ]:
def words(s):
    for ch in [',',':','!','_',';','-','.','?','/','"','\\n','\\']:
        s=s.lower().replace(ch,' ')
    return [x for x in s.split() if x]
counts=collections.Counter(w for v in train.values() for s in v['sentences'] for w in words(s))
vocab=[w for w,n in counts.items() if n>=2]+['<bos>','<eos>','UNK']
word_to_ix={w:i+1 for i,w in enumerate(vocab)}
vocab_path=REPO_DATA/'vocabulary_youcook2.json'
vocab_path.write_text(json.dumps({'word_to_ix':word_to_ix,'ix_to_word':{str(i):w for w,i in word_to_ix.items()}}))
import clip
sentences=[]; video_ids=[]
for vid,record in train.items():
    for sent in record['sentences']:
        sentences.append(sent.strip()); video_ids.append(vid)
model_clip,_=clip.load('ViT-L/14',device='cuda:0',jit=False); model_clip.eval()
encoded=[]
with torch.inference_mode():
    for j in range(0,len(sentences),64):
        tokens=clip.tokenize(sentences[j:j+64],truncate=True).to('cuda:0')
        encoded.append(model_clip.encode_text(tokens).float().cpu())
emb=torch.cat(encoded).numpy()
assert emb.shape==(len(sentences),768), emb.shape
for folder in ['clip','clip_arr_split']:
    out=REPO/'bank'/'yc2'/folder; out.mkdir(parents=True,exist_ok=True)
    import numpy as np
    np.save(out/'yc2_scene_sentences.npy',np.asarray(sentences,dtype=str))
    np.save(out/'yc2_clip_token_embeds.npy',emb)
    np.save(out/'yc2_scene_videoID.npy',np.asarray(video_ids,dtype=str))
print('CLIP memory entries:',len(sentences),emb.shape)
del model_clip,encoded,emb; torch.cuda.empty_cache()

## Runtime compatibility and CPU preflight

Keep the established single-GPU runtime fixes. Preflight checks quality targets, detached gradients, padded frames, head training/checkpoint restore, query-score alignment and actual ranking behavior. A tiny random GPT-2 test also validates the existing caption path; neither test downloads language-model weights.


In [ ]:
# Match the vendored Transformers dependency range; Kaggle currently ships huggingface-hub 1.x.
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface-hub<1.0'],check=True)

# Restore upstream single-GPU files if this cell was run before with the old 2-GPU adapter.
restore_files=[
    'train.py',
    'eval_utils_clip.py',
    'data/video_dataset.py',
    'ddvc/CaptioningHead/gpt2.py',
    'ddvc/ddvc_ret_encdec_clip_capQ.py',
    'transformers/dependency_versions_table.py',
]
subprocess.run(['git','-C',str(REPO),'checkout','--',*restore_files],check=True)
adapter_path=REPO/'kaggle_parallel.py'
if adapter_path.exists(): adapter_path.unlink()

# Replace the author's private tokenizer cache path with the public Hugging Face model id.
for rel in ['data/video_dataset.py','ddvc/CaptioningHead/gpt2.py']:
    p=REPO/rel
    s=p.read_text()
    s=s.replace("GPT2Tokenizer.from_pretrained('/DATA_DISK/plm_cache/gpt2')", "GPT2Tokenizer.from_pretrained('gpt2')")
    p.write_text(s)

# The vendored Transformers package predates Kaggle's Python 3.13 tokenizers wheel.
# Its tokenizer API is available, but the old upper-bound check rejects 0.23.x at import time.
dep=REPO/'transformers'/'dependency_versions_table.py'
s=dep.read_text()
old_req='"tokenizers": "tokenizers>=0.19,<0.20",'
new_req='"tokenizers": "tokenizers>=0.19,<0.24",'
if old_req in s:
    dep.write_text(s.replace(old_req,new_req))
elif new_req not in s:
    raise RuntimeError('Could not find vendored Transformers tokenizers version constraint')
# YouCook2 setup has no ActivityNet grounding GT file. Keep training/evaluation intact,
# but skip this unrelated grounding-only metric during validation.
eval_path=REPO/'eval_utils_clip.py'
eval_src=eval_path.read_text()
old_grounding="""    scores_g = eval_metrics_grounding(dvc_json_path + '.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
    aux_scores_g = eval_metrics_grounding(dvc_json_path + '_aux.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
"""
new_grounding="""    if getattr(opt, 'eval_enable_grounding', True):
        scores_g = eval_metrics_grounding(dvc_json_path + '.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
        aux_scores_g = eval_metrics_grounding(dvc_json_path + '_aux.grounding.json', gt_filename=opt.eval_gt_file_for_grounding)
    else:
        scores_g, aux_scores_g = {}, {}
"""
if old_grounding in eval_src:
    eval_src=eval_src.replace(old_grounding,new_grounding,1)
elif new_grounding not in eval_src:
    raise RuntimeError('Could not locate grounding metric calls in eval_utils_clip.py')
eval_path.write_text(eval_src)
print('Patched YouCook2 validation to skip ActivityNet-only grounding metrics.')

print('Restored upstream single-GPU training; applied Kaggle tokenizers compatibility check and public GPT-2 tokenizer.')

# Stop before training if the pinned source or local-mask wiring is missing.
assert 'raw_dvc_json_path' in (REPO/'eval_utils_clip.py').read_text()
assert 'event_visual_mask' in (REPO/'ddvc'/'CaptioningHead'/'gpt2.py').read_text()
subprocess.run([sys.executable, '-u', 'smoke_test_local_visual.py'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-u', 'smoke_test_quality.py'], cwd=REPO, check=True)
print('PASS: quality and caption CPU preflight. GPU training/validation follows.')


## Configure original DDVC + quality head

The original matcher/localization/caption losses are retained. The additional final-layer MSE learns best GT temporal IoU for all candidates. Detached inputs and separate clipping isolate the auxiliary head from the base network's gradient update. At inference, the head uses only predicted intervals, queries and video features.


In [ ]:
import yaml
opts=json.loads((REPO/'opts.json').read_text())
opts.update({
 'cfg_path':'','target_domain':'yc2','dataset':'yc2','bank_type':['yc2'],
 'train_caption_file':str(train_json.relative_to(REPO)),
 'val_caption_file':str(val_json.relative_to(REPO)),
 'gt_file_for_eval':[str(val_json.relative_to(REPO))],
 'gt_file_for_auc':[str(val_json.relative_to(REPO))],
 'gt_file_for_para_eval':[str(para_json.relative_to(REPO))],
 'visual_feature_folder':[str(features_path.relative_to(REPO))],
 'visual_feature_type':['clip'],'feature_dim':768,'frame_embedding_num':200,
 'dict_file':str(vocab_path.relative_to(REPO)),'vocab_size':len(word_to_ix),
 'bank_path':'bank','bank_folder':'clip','able_ret':True,
 'gpu_id':[0],'device':'cuda','batch_size':1,'batch_size_for_eval':1,
 'nthreads':0,'epoch':EPOCHS,'min_epoch_when_save':0,'save_checkpoint_every':1,
 'save_all_checkpoint':False,'save_dir':SAVE_DIR,'id':RUN_ID,
 'exp_name':f'{RUN_MODE}{len(train)}_e{EPOCHS}_{EXPERIMENT}_'+time.strftime('%Y%m%d_%H%M%S'),'start_from':RESUME_RUN_ID,'start_from_mode':'last','pretrain':None,
 'seed':SEED,'random_seed':False,'debug':False,'disable_tqdm':False,
 'train_proposal_sample_num':30,'gt_proposal_sample_num':10,
 'soft_k':20,'window_size':80,'ret_encoder':'multiscale_project',
 'max_caption_len':30,'gpt_beam_size':3,'num_queries':50,'decoder_query_ablation':False,
 'caption_decoder_type':'gpt2','caption_decoder_new':'none',
 'eval_enable_grounding':False,
 'with_box_refine':True,
 # Preserve original DDVC losses and global visual caption attention.
 'bbox_loss_coef':0.0,'set_cost_bbox':0.0,
 'local_visual_attention':False,
 'localization_quality':ENABLE_QUALITY,'quality_bins':8,
 'quality_loss_coef':QUALITY_LOSS_COEF,
 'quality_rank_weight':QUALITY_RANK_WEIGHT if ENABLE_QUALITY else 0.0,
 'gpt_layer':2,'add_cross_attention':True,
 # Make the existing reranking defaults explicit for reproducibility.
 'ec_alpha':1.0,'count_loss_coef':0.5,'eval_matching_score_weight':0.0,
 'caption_cost_chunk_size':16
})
(REPO/'cfgs').mkdir(exist_ok=True)
cfg=REPO/'cfgs'/'kaggle_yc2_quality_ranking.yml'
cfg.write_text(yaml.safe_dump(opts,sort_keys=False))
print(cfg, '| selected train/val:',len(train),len(val),'| vocab:',len(word_to_ix),'| epochs:',EPOCHS)
print('DDVC quality head:', ENABLE_QUALITY, '| ranking weight:', opts['quality_rank_weight'])
print("Reranking:", {k: opts[k] for k in ("ec_alpha", "count_loss_coef", "eval_matching_score_weight")})


## Train and evaluate

Validation uses the selected validation subset in smoke/subset mode and the complete feature-matched validation split in full mode. Training writes an unbuffered log and checkpoints each epoch. Set `RESUME_RUN_ID` in the first cell to the same run ID to continue from `model-last.pth`.


In [ ]:
run_env = os.environ.copy()
run_env.update({'CUDA_VISIBLE_DEVICES':'0','TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1'})
cmd = [sys.executable, '-u', 'train.py', '--cfg_path', str(cfg.relative_to(REPO))]
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
t0 = time.time()
with LOG_PATH.open('a', encoding='utf-8') as log:
    log.write(f"\n\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} | {RUN_MODE} | {len(train)} train / {len(val)} val | {EPOCHS} epochs ===\n")
    log.flush()
    proc = subprocess.Popen(cmd, cwd=REPO, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    returncode = proc.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, cmd)
print(f'Completed in {(time.time()-t0)/60:.1f} min; log: {LOG_PATH}')


## Summary / artifacts

In [ ]:
# Evaluate two rankings on the SAME best checkpoint's raw predictions (no GPU inference).
QUALITY_REPORT = None
run_root = REPO / SAVE_DIR
# train.py may append a descriptive suffix to the configured ID.
candidate_runs = sorted([p for p in run_root.glob(RUN_ID+'*') if (p/'info.json').exists()], key=lambda p:p.stat().st_mtime)
if not candidate_runs:
    raise FileNotFoundError(f'No completed validation info.json under {run_root}; inspect {LOG_PATH}')
comparison_run = run_root / RESUME_RUN_ID if RESUME_RUN_ID else candidate_runs[-1]
comparison_info = json.loads((comparison_run/'info.json').read_text())
comparison_best = comparison_info.get('best',{}).get('result_json_path')
if not comparison_best:
    raise RuntimeError('No best validation prediction recorded; inspect training log.')
comparison_final = Path(comparison_best)
if not comparison_final.is_absolute(): comparison_final = REPO / comparison_final
comparison_raw = comparison_final.with_suffix('.raw.json')
if ENABLE_QUALITY and QUALITY_RANK_WEIGHT > 0:
    QUALITY_REPORT = comparison_run / 'quality_ranking_comparison.json'
    comparison_cmd = [sys.executable, '-u', 'eval_quality_ablation.py', '--raw', str(comparison_raw),
                      '--config', str(cfg), '--output', str(QUALITY_REPORT)]
    with LOG_PATH.open('a',encoding='utf-8') as comparison_log:
        comparison_proc = subprocess.Popen(comparison_cmd,cwd=REPO,env=run_env,stdout=subprocess.PIPE,
                                          stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in comparison_proc.stdout:
            print(line,end=''); comparison_log.write(line); comparison_log.flush()
        comparison_rc = comparison_proc.wait()
    if comparison_rc: raise subprocess.CalledProcessError(comparison_rc,comparison_cmd)
    comparison_result = json.loads(QUALITY_REPORT.read_text())
    for label, metrics in comparison_result['metrics'].items():
        print(label, {k:metrics.get(k) for k in ['CIDEr','METEOR','soda_c','MetaScore','F1']})
    print('Changed selection:',comparison_result['changed_videos'],'/',comparison_result['videos'])
else:
    print('Paired quality comparison skipped: quality head or quality ranking disabled.')

run_root = REPO/SAVE_DIR
runs = sorted([p for p in run_root.iterdir() if p.is_dir()], key=lambda p:p.stat().st_mtime) if run_root.exists() else []
run = comparison_run
if run is None:
    print('No run directory was created. Check the training log:', LOG_PATH)
else:
    print('Run:', run)
    for name in ['model-best.pth','model-last.pth']:
        print(name + ':', 'present' if (run/name).exists() else 'missing')
    info_path = run/'info.json'
    if info_path.exists():
        info = json.loads(info_path.read_text(encoding='utf-8'))
        best, last = info.get('best',{}), info.get('last',{})
        print('Best epoch:', best.get('epoch'))
        print('Best validation score (METEOR + SODA_c):', best.get('best_val_score'))
        print('Best prediction:', best.get('result_json_path'))
        print('Last evaluated epoch:', last.get('epoch'))
        history = info.get('history',{}).get('val_result_history',{})
        for ep,row in history.items(): print('epoch',ep,row.get('eval_score',{}))
    else:
        print('No info.json yet: training did not complete a validation cycle; see log:', LOG_PATH)
    prediction_dir = run/'prediction'
    final_files = sorted(p for p in prediction_dir.glob('*.json') if '.raw.' not in p.name and not p.name.endswith(('.grounding.json', '.tal.json')))
    print('Final prediction files for visualization:', [str(p) for p in final_files][:20])
    print('Raw candidate files (diagnosis only):', [str(p) for p in prediction_dir.glob('*.raw.json')][:20])
    if info_path.exists() and best.get('result_json_path'):
        best_path = Path(best['result_json_path'])
        if not best_path.is_absolute():
            best_path = REPO/best_path
        raw_path = best_path.with_suffix('.raw.json')
        rerank_path = Path(str(raw_path)+f"_rerank_alpha{opts['ec_alpha']}_temp2.0" + (f"_quality{opts['quality_rank_weight']}" if opts['quality_rank_weight'] else '') + '.json')
        if best_path.exists():
            selected_output = json.loads(best_path.read_text())
            print('Best final JSON (use this for HTML):', best_path)
            print('Selected predictions/video:', selected_output.get('avg_proposal_num'))
            if raw_path.exists():
                print('Raw candidates/video:', json.loads(raw_path.read_text()).get('avg_proposal_num'))
            if rerank_path.exists():
                metric_input = json.loads(rerank_path.read_text())
                assert selected_output['results'] == metric_input['results'], 'Final JSON differs from the reranked metric input.'
                print('PASS: final JSON contains exactly the prediction list used by metrics.')


    # Small result bundle for analysis: predictions, actual references/config and logs.
    import shutil, zipfile
    bundle_dir = Path('/kaggle/working')/f'ddvc_{EXPERIMENT}_results'
    bundle_dir.mkdir(exist_ok=True)
    for name in ['info.json']:
        if (run/name).exists(): shutil.copy2(run/name, bundle_dir/name)
    for src, name in [(train_json, 'train.json'), (val_json, 'val.json'), (cfg, 'config.yml'), (LOG_PATH, 'train.log')]:
        if src.exists(): shutil.copy2(src, bundle_dir/name)
    if QUALITY_REPORT and QUALITY_REPORT.exists(): shutil.copy2(QUALITY_REPORT, bundle_dir/QUALITY_REPORT.name)
    (bundle_dir/'source_commit.txt').write_text(SOURCE_COMMIT+'\n')
    archive_path = bundle_dir.with_suffix('.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for item in bundle_dir.iterdir():
            if item.is_file(): archive.write(item, item.name)
        for item in (run/'prediction').glob('*.json'):
            archive.write(item, 'prediction/'+item.name)
    print('Download this result ZIP after training:', archive_path)


## Run modes / outputs

- Default subset: 30% of train and validation, 15 epochs. Smoke: 1%, 2 epochs. Full: feature-matched full data, 25 epochs.
- `ENABLE_QUALITY=False` runs the same-branch no-head baseline. Use a fresh run ID when changing head settings. Resume requires the same configuration and checkpoint.
- Run directory: `/kaggle/working/DDVC_quality_ranking/save_quality_ranking/<run-id>/`. `model-best.pth` and `model-last.pth` remain there. Save Kaggle output or download weights before the session ends.
- Log: `/kaggle/working/ddvc_quality_train.log` (or `ddvc_baseline_train.log`). Inspect `loss_quality`, `quality_mae`, `quality_mean`, `quality_target_mean`.
- The summary creates `/kaggle/working/ddvc_quality_results.zip`, including metrics, actual references/config, predictions, logs and `quality_ranking_comparison.json`. Checkpoints/features are excluded from this small ZIP.
- Use final selected prediction JSON for HTML. `*.raw.json` retains all candidates. Paired reranking JSON files identify original/quality selection explicitly.
- The paired report uses the same best checkpoint, selected by validation MetaScore with the configured ranking. It isolates ranking changes; a full-data, multi-seed comparison is needed to establish a reliable gain. Poor captions cannot be repaired by localization-quality ranking alone.
